# MAP5935 - Statistical Learning (Chapter 7 - Moving Beyond Linearity)

**Prof. Christian Jäkel**

https://www.statlearning.com/

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.pyplot import subplots
import statsmodels.api as sm

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer, make_column_selector as selector
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.linear_model import RidgeCV
from sklearn.linear_model import LassoCV
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression

from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

## *Conceptual Exercises*

### (3) Suppose we fit a curve with basis functions $b_1(X) = X$, $b_2(X) = (X − 1)^2 I_{(X\geq 1)}$. (Note that $I_{(X\geq 1)}$ equals $1$ for $X\geq 1$ and $0$ otherwise.) We fit the linear regression model.
### $$Y=\beta_0 + \beta_1 b_1(X) + \beta_2 b_2(X) + \epsilon$$
### and obtain coefficient estimates $\hat\beta_0=1$, $\hat\beta_1=1$, $\hat\beta_2=-2$. Sketch the estimated curve between $X = −2$ and $X = 2$. Note the intercepts, slopes, and other relevant information.

**Solution**: We fit a model with basis functions $b_1(X)=X$, $b_2(X)=(X-1)^2\,\mathbb{1}\{X\ge 1\}$.

Given $\hat\beta_0=1,\ \hat\beta_1=1,\ \hat\beta_2=-2$, the fitted function is

$$
\widehat{Y}(x)=1 + x - 2\,(x-1)^2\,\mathbb{1}\{x\ge 1\}.
$$
Piecewise:
$$
\widehat{Y}(x)=
\begin{cases}
1+x, & x<1,\\[4pt]
1+x-2(x-1)^2, & x\ge 1.
\end{cases}
$$

**Key properties (for $x\in[-2,2]$):**
- **Intercept:** $\widehat{Y}(0)=1$.
- **At the knot $x=1$:** $\widehat{Y}(1^-)=\widehat{Y}(1^+)=2$ (continuous).
- **Slope:**
  - $x<1:\ \widehat{Y}'(x)=1$.
  - $x\ge 1:\ \widehat{Y}'(x)=1-4(x-1)=5-4x$.
  - Right‐slope at $x=1$ is $1$ → continuously differentiable at the knot.
- **Curvature for $x\ge 1$:** concave down (quadratic coefficient $-2$).
- **Local maximum (for $x\ge 1$):**
  $$
  \widehat{Y}'(x)=0 \Rightarrow x^\star=\tfrac{5}{4}=1.25,\quad
  \widehat{Y}(1.25)=1+1.25-2(0.25)^2=2.125.
  $$
- **Endpoint values:** $\widehat{Y}(-2)=-1$, $\widehat{Y}(2)=1$.

**Sketch instructions:**
1. Draw the line $y=1+x$ for $x<1$ (through $(-2,-1)$, $(0,1)$, reaching $(1,2)$).
2. For $x\ge 1$, continue with $y=1+x-2(x-1)^2$: smooth at $(1,2)$, peaks at $(1.25,\,2.125)$, then decreases to $(2,1)$.

In [3]:
# --- 1) Synthetic data ---
rng = np.random.default_rng(42)
n = 400
X = rng.uniform(-2, 2, size=n)

# basis functions
b1 = X
b2 = (X - 1.0)**2 * (X >= 1.0)  # (X-1)^2 * I(X>=1)

# true coefficients
beta0, beta1, beta2 = 1.0, 1.0, -2.0

# noise
sigma = 0.4
eps = rng.normal(0.0, sigma, size=n)

# response
Y = beta0 + beta1*b1 + beta2*b2 + eps

# put into a DataFrame (handy for inspection)
df = pd.DataFrame({"Y": Y, "X": X, "b1": b1, "b2": b2})

# --- 2) Fit linear regression with the basis functions ---
X_design = sm.add_constant(df[["b1", "b2"]])  # adds intercept
model = sm.OLS(df["Y"], X_design).fit()

print(model.summary())

# --- 3) Compare estimated vs. true coefficients ---
est = model.params
comparison = pd.DataFrame({
    "parameter": ["Intercept", "beta1 (X)", "beta2 ((X-1)^2 I{X>=1})"],
    "true": [beta0, beta1, beta2],
    "estimated": [est["const"], est["b1"], est["b2"]],
})
print("\nEstimated vs. true coefficients:")
print(comparison.to_string(index=False))

# Optional: in-sample RMSE
rmse = np.sqrt(np.mean(model.resid**2))
print(f"\nIn-sample RMSE: {rmse:.4f}")

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.857
Model:                            OLS   Adj. R-squared:                  0.857
Method:                 Least Squares   F-statistic:                     1193.
Date:                Thu, 09 Oct 2025   Prob (F-statistic):          1.41e-168
Time:                        00:16:03   Log-Likelihood:                -205.14
No. Observations:                 400   AIC:                             416.3
Df Residuals:                     397   BIC:                             428.2
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.9678      0.023     42.918      0.0

## *Applied Exercises*

### (10) This question relates to the `College` data set.

### (a) Split the data into a training set and a test set. Using out-of-state tuition as the response and the other variables as the predictors, perform forward stepwise selection on the training set in order to identify a satisfactory model that uses just a subset of the predictors.

In [4]:
College = pd.read_csv("../Data/College.csv")

In [5]:
College.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 777 entries, 0 to 776
Data columns (total 19 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Unnamed: 0   777 non-null    object 
 1   Private      777 non-null    object 
 2   Apps         777 non-null    int64  
 3   Accept       777 non-null    int64  
 4   Enroll       777 non-null    int64  
 5   Top10perc    777 non-null    int64  
 6   Top25perc    777 non-null    int64  
 7   F.Undergrad  777 non-null    int64  
 8   P.Undergrad  777 non-null    int64  
 9   Outstate     777 non-null    int64  
 10  Room.Board   777 non-null    int64  
 11  Books        777 non-null    int64  
 12  Personal     777 non-null    int64  
 13  PhD          777 non-null    int64  
 14  Terminal     777 non-null    int64  
 15  S.F.Ratio    777 non-null    float64
 16  perc.alumni  777 non-null    int64  
 17  Expend       777 non-null    int64  
 18  Grad.Rate    777 non-null    int64  
dtypes: float

In [6]:
College.head()

,Unnamed: 0,Private,Apps,Accept,Enroll,Top10perc,Top25perc,F.Undergrad,P.Undergrad,Outstate,Room.Board,Books,Personal,PhD,Terminal,S.F.Ratio,perc.alumni,Expend,Grad.Rate
0,Abilene Christian University,Yes,1660,1232,721,23,52,2885,537,7440,3300,450,2200,70,78,18.1,12,7041,60
1,Adelphi University,Yes,2186,1924,512,16,29,2683,1227,12280,6450,750,1500,29,30,12.2,16,10527,56
2,Adrian College,Yes,1428,1097,336,22,50,1036,99,11250,3750,400,1165,53,66,12.9,30,8735,54
3,Agnes Scott College,Yes,417,349,137,60,89,510,63,12960,5450,450,875,92,97,7.7,37,19016,59
4,Alaska Pacific University,Yes,193,146,55,16,44,249,869,7560,4120,800,1500,76,72,11.9,2,10922,15


We’ll fit a logistic regression using **income** and **balance** to predict **default**.
The response needs to be binary, so we map `"Yes"→1`, `"No"→0`.

In [7]:
# Get a copy of the DataFrame
df = College.copy()

# Drop non-informative ID column if present
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

# Define target and features
y = df["Apps"]
X = df.drop(columns=["Apps"])

# Optional stratification by the `Private` flag (helps keep balance)
stratifier = X["Private"] if "Private" in X.columns else None

# Split: 80% train / 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=stratifier
)

# Quick checks
print("Shapes:")
print("  X_train:", X_train.shape, "  y_train:", y_train.shape)
print("  X_test :", X_test.shape,  "  y_test :", y_test.shape)

Shapes:
  X_train: (621, 17)   y_train: (621,)
  X_test : (156, 17)   y_test : (156,)


### (b) Fit a GAM on the training data, using out-of-state tuition as the response and the features selected in the previous step as the predictors. Plot the results, and explain your findings.

### (c) Evaluate the model obtained on the test set, and explain the results obtained.

### (d) For which variables, if any, is there evidence of a non-linear relationship with the response?